# Mimosa Gravel — Lakehouse Data Generator

**Fictional European Gravel Bike Reseller — South West France**  
Generates 5 years of synthetic transactional data (Feb 2021 → July 2026) with realistic seasonality and a growing market CAGR of ~28%.

### Tables generated
| Layer | Table | Approx. rows |
|---|---|---|
| Dim | `dim_date` | 1 827 |
| Dim | `dim_product` (SCD2) | ~800 |
| Dim | `dim_customer` (SCD2) | ~160 000 |
| Dim | `dim_store` | 2 |
| Dim | `dim_supplier` | ~15 |
| Dim | `dim_campaign` | ~60 |
| Dim | `dim_return_reason` | 6 |
| Fact | `fact_sales` | **~500 M** |
| Fact | `fact_returns` | ~20 M |
| Fact | `fact_inventory` | ~100 M |
| Fact | `fact_marketing` | ~200 K |

> **Platform**: Databricks / Microsoft Fabric (PySpark)  
> Set `BASE_PATH` in the **Config** cell below before running.

## 0 · Config & Parameters

In [ ]:
# ── USER CONFIG ─────────────────────────────────────────────────────────────
# Set the base Delta Lake path. Examples:
#   Fabric  : 'abfss://7a35cc4f-f866-4a81-b753-a6c63e722407@onelake.dfs.fabric.microsoft.com/c9f8d73d-c9d4-442a-a1e4-3ba0a5f881c8/Files'
#   DBFS    : '/mnt/lakehouse/mimosa_gravel'
#   Local   : '/tmp/mimosa_gravel'
BASE_PATH   = 'abfss://7a35cc4f-f866-4a81-b753-a6c63e722407@onelake.dfs.fabric.microsoft.com/c9f8d73d-c9d4-442a-a1e4-3ba0a5f881c8/Files'          # <── change me
DATABASE    = 'mimosa_gravel'
RANDOM_SEED = 42

# ── DATE RANGE ──────────────────────────────────────────────────────────────
START_DATE  = '2021-02-01'
END_DATE    = '2026-07-10'

# ── VOLUME CONTROL ──────────────────────────────────────────────────────────
# Base order-lines in year-1 (~2021). Grows at CAGR below.
# Increase BASE_LINES_Y1 to push toward 500 M+ total. At 22 M it yields ~180 M;
# set to 1 M. Adjust to cluster memory.
BASE_LINES_Y1 = 1000000
CAGR          = 0.28          # 28 % yearly growth

# ── DIMS SIZES ──────────────────────────────────────────────────────────────
N_CUSTOMERS   = 80000
N_SKU         = 400

print('Config loaded.')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 8, Finished, Available, Finished, False)

Config loaded.


## 1 · Imports & Spark Setup

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

import math, random
from datetime import date, timedelta
from itertools import product as iproduct

try:
    from faker import Faker
    from faker.providers import address, person, internet, company
    FAKER_OK = True
except ImportError:
    print('Faker not installed — installing now...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faker', '-q'])
    from faker import Faker
    FAKER_OK = True

# ── Spark session (already exists on Databricks/Fabric; creates one locally)
spark = SparkSession.builder \
    .appName('MimosaGravel_DataGen') \
    .config('spark.sql.session.timeZone', 'Europe/Paris') \
    .getOrCreate()

spark.conf.set('spark.sql.shuffle.partitions', '400')
spark.sql(f'CREATE DATABASE IF NOT EXISTS {DATABASE}')
spark.sql(f'USE {DATABASE}')

random.seed(RANDOM_SEED)
print(f'Spark {spark.version} ready | Database: {DATABASE}')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 10, Finished, Available, Finished, False)

Faker not installed — installing now...
Spark 3.5.5.5.4.20260109.1 ready | Database: mimosa_gravel


## 2 · `dim_date`

In [ ]:
# ── Gravel race events (approximate real-world dates) ──────────────────────
RACE_EVENTS = {
    # (month, day): event_name
    (4, 24): 'Traka 360',
    (6,  3): 'Unbound Gravel',
    (8, 14): 'Badlands Gravel',
    (10, 5): 'Gravel World Champs',
    (3, 15): 'Strade Bianche'
}

# Extend event window ±7 days
EVENT_DATES = set()
for (m, d), _ in RACE_EVENTS.items():
    for yr in range(2021, 2027):
        try:
            ev = date(yr, m, d)
            for delta in range(-7, 8):
                EVENT_DATES.add(str(ev + timedelta(days=delta)))
        except ValueError:
            pass

# ── Build date rows in Python, then parallelize ────────────────────────────
start = date(2021, 2, 1)
end   = date(2026, 7, 10)
date_rows = []
d = start
while d <= end:
    ds = str(d)
    m  = d.month
    if m in (3, 4, 5):   season, base_weight = 'Spring', 1.40
    elif m in (6, 7, 8): season, base_weight = 'Summer', 1.10
    elif m in (9, 10):   season, base_weight = 'Autumn', 0.60
    else:                season, base_weight = 'Winter', 1.30  # Nov-Feb

    # Extra boost in peak months
    if m == 4:  base_weight += 0.40
    if m == 5:  base_weight += 0.20
    if m == 12: base_weight += 0.40

    is_race = ds in EVENT_DATES
    if is_race: base_weight = min(base_weight * 1.5, 3.0)

    date_rows.append((
        int(d.strftime('%Y%m%d')),   # date_key
        ds,                          # full_date
        d.year,
        ((d.month - 1) // 3) + 1,   # quarter
        d.month,
        d.isocalendar()[1],          # iso_week
        d.day,
        d.strftime('%A'),            # day_name
        d.isoweekday(),              # day_of_week  (1=Mon)
        d.isoweekday() >= 6,         # is_weekend
        season,
        is_race,                     # is_race_event_window
        round(base_weight, 3),       # month_demand_weight
        int(d.strftime('%Y%m')),     # year_month
    ))
    d += timedelta(days=1)

dim_date_schema = T.StructType([
    T.StructField('date_key',              T.IntegerType(),   False),
    T.StructField('full_date',             T.StringType(),    False),
    T.StructField('year',                  T.IntegerType(),   False),
    T.StructField('quarter',               T.IntegerType(),   False),
    T.StructField('month',                 T.IntegerType(),   False),
    T.StructField('iso_week',              T.IntegerType(),   False),
    T.StructField('day',                   T.IntegerType(),   False),
    T.StructField('day_name',              T.StringType(),    False),
    T.StructField('day_of_week',           T.IntegerType(),   False),
    T.StructField('is_weekend',            T.BooleanType(),   False),
    T.StructField('season',                T.StringType(),    False),
    T.StructField('is_race_event_window',  T.BooleanType(),   False),
    T.StructField('month_demand_weight',   T.FloatType(),     False),
    T.StructField('year_month',            T.IntegerType(),   False),
])

dim_date = spark.createDataFrame(date_rows, schema=dim_date_schema)
dim_date.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .saveAsTable(f'{DATABASE}.dim_date')

print(f'dim_date: {dim_date.count():,} rows')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 11, Finished, Available, Finished, False)

dim_date: 1,853 rows


## 3 · `dim_supplier`

In [9]:
# ── Component & own-brand suppliers ───────────────────────────────────────
supplier_rows = [
    # (supplier_key, supplier_name, category, country, currency, lead_time_days, payment_terms, is_own_brand)
    (1,  'Shimano Europe GmbH',         'Components',   'DE', 'EUR', 21, 'Net 60', False),
    (2,  'SRAM LLC Europe',             'Components',   'NL', 'EUR', 28, 'Net 45', False),
    (3,  'Campagnolo S.r.l.',           'Components',   'IT', 'EUR', 35, 'Net 60', False),
    (4,  'DT Swiss AG',                 'Wheels',       'CH', 'CHF', 30, 'Net 45', False),
    (5,  'Hunt Bike Wheels',            'Wheels',       'GB', 'GBP', 25, 'Net 30', False),
    (6,  'Zipp / SRAM',                 'Wheels',       'IE', 'EUR', 28, 'Net 45', False),
    (7,  'Fizik S.p.A.',                'Saddles',      'IT', 'EUR', 20, 'Net 30', False),
    (8,  'Ritchey Design',              'Handlebars',   'US', 'EUR', 40, 'Net 60', False),
    (9,  'Castelli Cycling',            'Apparel',      'IT', 'EUR', 45, 'Net 60', False),
    (10, 'Assos of Switzerland',        'Apparel',      'CH', 'CHF', 40, 'Net 45', False),
    (11, 'Café du Cycliste',            'Apparel',      'FR', 'EUR', 15, 'Net 30', False),
    (12, 'Mimosa Gravel Manufacture',   'Bikes',        'FR', 'EUR',  7, 'N/A',    True),
    (13, 'Mimosa Gravel Apparel',       'Apparel',      'FR', 'EUR',  7, 'N/A',    True),
    (14, 'Vittoria Industries',         'Tyres',        'IT', 'EUR', 25, 'Net 45', False),
    (15, 'Pirelli Cycling',             'Tyres',        'IT', 'EUR', 25, 'Net 45', False),
]

supplier_schema = T.StructType([
    T.StructField('supplier_key',    T.IntegerType(), False),
    T.StructField('supplier_name',   T.StringType(),  False),
    T.StructField('category',        T.StringType(),  False),
    T.StructField('country',         T.StringType(),  False),
    T.StructField('currency',        T.StringType(),  False),
    T.StructField('lead_time_days',  T.IntegerType(), False),
    T.StructField('payment_terms',   T.StringType(),  False),
    T.StructField('is_own_brand',    T.BooleanType(), False),
])

dim_supplier = spark.createDataFrame(supplier_rows, schema=supplier_schema)
dim_supplier.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .saveAsTable(f'{DATABASE}.dim_supplier')
print(f'dim_supplier: {dim_supplier.count()} rows')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 12, Finished, Available, Finished, False)

dim_supplier: 15 rows


## 4 · `dim_product` (SCD2)

In [10]:
import uuid

rng = random.Random(RANDOM_SEED + 1)

# ── Product catalog blueprint ─────────────────────────────────────────────
BIKE_MODELS = [
    ('MG-E100', 'Mimosa Antelope', 'Complete Bike', 'Entry', 2199, 12, True),
    ('MG-M200', 'Mimosa Fennec', 'Complete Bike', 'Mid-Range', 3299, 12, True),
    ('MG-P300', 'Mimosa Camel', 'Complete Bike', 'Pro', 5499, 12, True),
    ('MG-FS10', 'Mimosa Antelope Frameset', 'Frameset', 'Entry', 1199, 12, True),
    ('MG-FS20', 'Mimosa Fennec Frameset', 'Frameset', 'Mid-Range', 1599, 12, True),
    ('MG-FS30', 'Mimosa Camel Frameset', 'Frameset', 'Pro', 2199, 12, True),
]

COMPONENT_TEMPLATES = [
    # (prefix, name_tmpl, subcategory, price_range, supplier_key, qty_stock_base)
    ('SHIM-GRX8', 'Shimano GRX 810 Groupset 1x11', 'Groupset', (450,  520), 1, 30),
    ('SHIM-GRX6', 'Shimano GRX 600 Groupset 2x10', 'Groupset', (320,  380), 1, 40),
    ('SRAM-RIV',  'SRAM Rival AXS XPLR 1x12',      'Groupset', (680,  760), 2, 25),
    ('SRAM-FRC',  'SRAM Force AXS XPLR 1x12',      'Groupset', (1050,1150), 2, 15),
    ('SRAM-RED',  'SRAM Red AXS XPLR 1x12',        'Groupset', (2100,2300), 2, 10),
    ('CAMP-EKR',  'Campagnolo Ekar 1x13',           'Groupset', (1200,1400), 3, 12),
    ('DT-232E',   'DT Swiss 232 Enduro Wheel 700c', 'Wheel Set',(680,  750), 4, 20),
    ('HNT-ADV',   'Hunt 4 Season Gravel Wheelset',  'Wheel Set',(420,  460), 5, 25),
    ('ZPP-303F',  'Zipp 303 Firecrest Disc',        'Wheel Set',(1600,1750), 6, 8),
    ('FIZ-ARI',   'Fizik Ares R5 Shoes 41-47',      'Shoes',    (220,  260), 7, 50),
    ('FIZ-SAD',   'Fizik Tempo Argo R3 Saddle',     'Saddle',   (130,  160), 7, 60),
    ('RIT-VEN',   'Ritchey Venturemax Handlebar',   'Handlebar',(85,   110), 8, 70),
    ('VIT-CRS',   'Vittoria Corsa Pro Gravel 700x38','Tyre',     (58,    72), 14, 120),
    ('PIR-CIN',   'Pirelli Cinturato Gravel M 40mm', 'Tyre',    (52,    65), 15, 100),
]

APPAREL_TEMPLATES = [
    ('MGA-JRS', 'Mimosa Gravel Jersey SS',       'Jersey',  (95,  115), 13, 80, True),
    ('MGA-JSL', 'Mimosa Gravel Jersey LS',       'Jersey',  (115, 135), 13, 60, True),
    ('MGA-BIB', 'Mimosa Gravel Bib Short',       'Bibs',    (145, 175), 13, 70, True),
    ('MGA-GIL', 'Mimosa Gravel Gilet',           'Gilet',   (120, 145), 13, 50, True),
    ('MGA-CAP', 'Mimosa Gravel Cap',             'Cap',     (28,   35), 13, 150,True),
    ('MGA-GLV', 'Mimosa Gravel Gloves',          'Gloves',  (40,   55), 13, 100,True),
    ('CAS-PRO', 'Castelli Perfetto RoS Jersey',  'Jersey',  (180, 210), 9,  40, False),
    ('ASS-EQP', 'Assos Equipe RS S9 Bib',        'Bibs',    (320, 370), 10, 25, False),
    ('CDC-MRS', 'Café du Cycliste Marinière J.',  'Jersey',  (140, 165), 11, 30, False),
    ('MGA-HLM', 'Giro Manifest Spherical Helmet','Helmet',  (260, 310), 13, 35, True),
    ('MGA-BAG', 'Apidura Backcountry Saddle Bag','Bag',     (90,  120), 13, 45, True),
    ('MGA-LGT', 'Lezyne Mega Drive 1800 Light',  'Light',   (75,   95), 13, 55, True),
]

SERVICE_TEMPLATES = [
    ('SVC-BLD', 'Complete Bike Build',     'Workshop', (120, 180), 12, 0),
    ('SVC-FIT', 'Professional Bike Fit',   'Workshop', (180, 250), 12, 0),
    ('SVC-TUN', 'Full Service & Tune-Up',  'Workshop', (80,  120), 12, 0),
    ('SVC-WHL', 'Wheel Truing & Service',  'Workshop', (35,   60), 12, 0),
]

# SIZE variants for apparel
SIZES = ['XS', 'S', 'M', 'L', 'XL', 'XXL']

def make_products():
    rows = []
    sk = 1  # surrogate key counter

    def valid_to(valid_from_str):
        """SCD2 expiry — pick a change date ~18-30m after valid_from, max end-date."""
        vf = date.fromisoformat(valid_from_str)
        months = rng.randint(18, 36)
        new_d = min(date(vf.year + months // 12, ((vf.month - 1 + months) % 12) + 1, 1),
                    date(2026, 2, 27))
        return str(new_d)

    def add(product_nk, name, category, subcategory, unit_price, supplier_key,
            is_own_brand, lifecycle, valid_from, valid_to_val, is_current, size=None):
        nonlocal sk
        sku_suffix = f'-{size}' if size else ''
        rows.append((
            sk, product_nk + sku_suffix, name + (f' {size}' if size else ''),
            category, subcategory, float(unit_price), supplier_key,
            is_own_brand, lifecycle, size,
            valid_from, valid_to_val, is_current
        ))
        sk += 1

    # ── Bikes (no size variants)
    for code, name, cat, sub, price, sup, own in BIKE_MODELS:
        vf1 = '2021-02-01'
        vt1 = valid_to(vf1)
        add(code, name, cat, sub, price, sup, own, 'Active',       vf1, vt1, False)
        add(code, name, cat, sub, round(price * rng.uniform(1.03, 1.10)), sup, own, 'Active', vt1, '9999-12-31', True)

    # ── Components
    for code, name, sub, (plo, phi), sup, _ in COMPONENT_TEMPLATES:
        price_v1 = rng.randint(plo, phi)
        price_v2 = round(price_v1 * rng.uniform(1.04, 1.12))
        vf1 = '2021-02-01'
        vt1 = valid_to(vf1)
        add(code, name, 'Component', sub, price_v1, sup, False, 'Active', vf1, vt1, False)
        add(code, name, 'Component', sub, price_v2, sup, False, 'Active', vt1, '9999-12-31', True)

    # ── Apparel (with size variants)
    for code, name, sub, (plo, phi), sup, _, own in APPAREL_TEMPLATES:
        for size in SIZES:
            price_v1 = rng.randint(plo, phi)
            price_v2 = round(price_v1 * rng.uniform(1.05, 1.15))
            vf1 = '2021-02-01'
            vt1 = valid_to(vf1)
            add(code, name, 'Apparel', sub, price_v1, sup, own, 'Active', vf1, vt1, False, size)
            add(code, name, 'Apparel', sub, price_v2, sup, own, 'Active', vt1, '9999-12-31', True, size)

    # ── Services (no size, no SCD2 version change)
    for code, name, sub, (plo, phi), sup, _ in SERVICE_TEMPLATES:
        price = rng.randint(plo, phi)
        add(code, name, 'Service', sub, price, sup, True, 'Active', '2021-02-01', '9999-12-31', True)

    return rows

product_rows = make_products()

product_schema = T.StructType([
    T.StructField('product_key',    T.IntegerType(),   False),
    T.StructField('product_nk',     T.StringType(),    False),
    T.StructField('product_name',   T.StringType(),    False),
    T.StructField('category',       T.StringType(),    False),
    T.StructField('subcategory',    T.StringType(),    False),
    T.StructField('unit_price_eur', T.DoubleType(),    False),
    T.StructField('supplier_key',   T.IntegerType(),   False),
    T.StructField('is_own_brand',   T.BooleanType(),   False),
    T.StructField('lifecycle',      T.StringType(),    False),
    T.StructField('size',           T.StringType(),    True),
    T.StructField('valid_from',     T.StringType(),    False),
    T.StructField('valid_to',       T.StringType(),    False),
    T.StructField('is_current',     T.BooleanType(),   False),
])

dim_product = spark.createDataFrame(product_rows, schema=product_schema)
dim_product.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .saveAsTable(f'{DATABASE}.dim_product')
print(f'dim_product: {dim_product.count():,} rows (including SCD2 versions)')
print(f'  Distinct SKUs (current only): {dim_product.filter("is_current").select("product_nk").distinct().count():,}')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 13, Finished, Available, Finished, False)

dim_product: 188 rows (including SCD2 versions)
  Distinct SKUs (current only): 96


## 5 · `dim_customer` (SCD2)

In [11]:
# ── Fake profile generation (runs on driver, parallelised by broadcast) ───

COUNTRY_DIST = [
    # (country_code, locale, weight, region_hint)
    ('FR', 'fr_FR',  0.55, 'France'),
    ('GB', 'en_GB',  0.10, 'United Kingdom'),
    ('DE', 'de_DE',  0.08, 'Germany'),
    ('BE', 'fr_BE',  0.04, 'Belgium'),
    ('NL', 'nl_NL',  0.04, 'Netherlands'),
    ('ES', 'es_ES',  0.04, 'Spain'),
    ('IT', 'it_IT',  0.04, 'Italy'),
    ('CH', 'de_CH',  0.03, 'Switzerland'),
    ('SE', 'sv_SE',  0.02, 'Sweden'),
    ('NO', 'no_NO',  0.02, 'Norway'),
    ('PL', 'pl_PL',  0.02, 'Poland'),
    ('PT', 'pt_PT',  0.01, 'Portugal'),
    ('DK', 'da_DK',  0.01, 'Denmark'),
]

LOYALTY_TIERS = ['Bronze', 'Silver', 'Gold']
SEGMENTS      = ['Amateur', 'Enthusiast', 'Club Member', 'Professional', 'Commuter']

def generate_customers(n, seed):
    rng_c = random.Random(seed)
    countries   = [c[0] for c in COUNTRY_DIST]
    locales_map = {c[0]: c[1] for c in COUNTRY_DIST}
    weights     = [c[2] for c in COUNTRY_DIST]

    fakers = {c: Faker(loc) for c, loc in locales_map.items()}
    Faker.seed(seed)

    rows = []
    sk = 1

    join_start = date(2021, 2, 1)
    join_end   = date(2025, 6, 1)

    for i in range(n):
        country = rng_c.choices(countries, weights=weights, k=1)[0]
        fk = fakers[country]

        gender   = rng_c.choice(['M', 'F'])
        first    = fk.first_name_male() if gender == 'M' else fk.first_name_female()
        last     = fk.last_name()
        email    = fk.email()
        city     = fk.city()
        postcode = fk.postcode()
        segment  = rng_c.choices(SEGMENTS, weights=[35,30,20,5,10], k=1)[0]
        tier_v1  = 'Bronze'
        tier_v2  = rng_c.choice(['Silver', 'Gold']) if segment in ('Enthusiast','Professional') else 'Bronze'

        # Join date — random in range
        delta_days = (join_end - join_start).days
        jd  = join_start + timedelta(days=rng_c.randint(0, delta_days))
        # SCD2 change date
        change_days = rng_c.randint(180, 900)
        cd  = min(jd + timedelta(days=change_days), date(2026, 2, 27))

        # Version 1
        rows.append((
            sk, i + 1, first, last, email, gender, country, city, postcode,
            segment, tier_v1, str(jd), str(cd), False
        ))
        sk += 1
        # Version 2
        rows.append((
            sk, i + 1, first, last, email, gender, country, city, postcode,
            segment, tier_v2, str(cd), '9999-12-31', True
        ))
        sk += 1

    return rows

print('Generating customer profiles...')
customer_rows = generate_customers(N_CUSTOMERS, RANDOM_SEED + 2)

customer_schema = T.StructType([
    T.StructField('customer_key',    T.IntegerType(), False),
    T.StructField('customer_nk',     T.IntegerType(), False),
    T.StructField('first_name',      T.StringType(),  False),
    T.StructField('last_name',       T.StringType(),  False),
    T.StructField('email',           T.StringType(),  False),
    T.StructField('gender',          T.StringType(),  False),
    T.StructField('country',         T.StringType(),  False),
    T.StructField('city',            T.StringType(),  False),
    T.StructField('postcode',        T.StringType(),  True),
    T.StructField('segment',         T.StringType(),  False),
    T.StructField('loyalty_tier',    T.StringType(),  False),
    T.StructField('valid_from',      T.StringType(),  False),
    T.StructField('valid_to',        T.StringType(),  False),
    T.StructField('is_current',      T.BooleanType(), False),
])

dim_customer = spark.createDataFrame(customer_rows, schema=customer_schema)
dim_customer.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .saveAsTable(f'{DATABASE}.dim_customer')
print(f'dim_customer: {dim_customer.count():,} rows (×2 SCD2 versions per customer)')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 14, Finished, Available, Finished, False)

Generating customer profiles...
dim_customer: 160,000 rows (×2 SCD2 versions per customer)


## 6 · `dim_store`, `dim_campaign`, `dim_return_reason`

In [12]:
# ── dim_store ─────────────────────────────────────────────────────────────
store_rows = [
    (1, 'Mimosa Gravel – Bayonne',  'Physical', '12 Rue du Rendez-vous', 'Bayonne', '64100', 'FR', 'Nouvelle-Aquitaine', '2020-03-15', None),
    (2, 'Mimosa Gravel – Gijona',  'Physical', '12 Carrer dels Cardels', 'Gijona', None, 'ES', 'Cataluña', '2022-04-12', None),
    (3, 'Mimosa Gravel – Online',    'Online',   'www.mimosagravel.fr', 'Paris', None, 'FR', 'Online', '2021-02-01', None),
]
store_schema = T.StructType([
    T.StructField('store_key',       T.IntegerType(), False),
    T.StructField('store_name',      T.StringType(),  False),
    T.StructField('store_type',      T.StringType(),  False),
    T.StructField('address',         T.StringType(),  True),
    T.StructField('city',            T.StringType(),  True),
    T.StructField('postcode',        T.StringType(),  True),
    T.StructField('country',         T.StringType(),  False),
    T.StructField('region',          T.StringType(),  True),
    T.StructField('opening_date',    T.StringType(),  False),
    T.StructField('closing_date',    T.StringType(),  True),
])
dim_store = spark.createDataFrame(store_rows, schema=store_schema)
dim_store.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .saveAsTable(f'{DATABASE}.dim_store')
print(f'dim_store: {dim_store.count()} rows')

# ── dim_return_reason ─────────────────────────────────────────────────────
rr_rows = [
    (1, 'Wrong size',            'Customer error'),
    (2, 'Defective product',     'Quality issue'),
    (3, 'Changed mind',          'Customer error'),
    (4, 'Damaged in transit',    'Logistics'),
    (5, 'Compatibility issue',   'Customer error'),
    (6, 'Duplicate order',       'Customer error'),
]
rr_schema = T.StructType([
    T.StructField('return_reason_key',  T.IntegerType(), False),
    T.StructField('reason',             T.StringType(),  False),
    T.StructField('reason_category',    T.StringType(),  False),
])
dim_return_reason = spark.createDataFrame(rr_rows, schema=rr_schema)
dim_return_reason.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .saveAsTable(f'{DATABASE}.dim_return_reason')
print(f'dim_return_reason: {dim_return_reason.count()} rows')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 15, Finished, Available, Finished, False)

dim_store: 3 rows
dim_return_reason: 6 rows


In [13]:
# ── dim_campaign ──────────────────────────────────────────────────────────
from datetime import date

campaign_templates = [
    # (name, channel, month_start, day_start, duration_days, budget_eur)
    ('Spring Launch',         'email',        3,  1,  30, 3000),
    ('Traka Race Promo',      'paid_social',  4, 17,  14, 5000),
    ('Unbound Inspiration',   'influencer',   5, 26,  14, 8000),
    ('Summer Adventure',      'display',      6, 15,  45, 4000),
    ('Badlands Special',      'paid_social',  8,  7,  14, 6000),
    ('Back to Trails',        'email',        9,  1,  21, 2500),
    ('Gravel Worlds Week',    'influencer',  10,  1,  14, 7000),
    ('Black Friday',          'display',     11, 22,  10,10000),
    ('Cyber Week',            'email',       11, 28,   7, 4000),
    ('Christmas Gift Guide',   'paid_social', 12,  1,  24, 6000),
    ('New Year New Season',   'email',        1,  2,  14, 2000),
    ('Loyalty Re-engagement', 'email',        2,  1,  21, 1500),
]

campaign_rows = []
ck = 1
for yr in range(2021, 2027):
    for name, channel, ms, ds, dur, budget in campaign_templates:
        try:
            sd = date(yr, ms, ds)
            ed = sd + timedelta(days=dur - 1)
            if sd > date(2026, 2, 27):
                continue
            ed = min(ed, date(2026, 2, 27))
            campaign_rows.append((
                ck, f'{name} {yr}', channel,
                str(sd), str(ed),
                float(budget * (1 + (yr - 2021) * 0.12)),  # budget grows YoY
                yr
            ))
            ck += 1
        except ValueError:
            pass  # Feb 30, etc.

campaign_schema = T.StructType([
    T.StructField('campaign_key',   T.IntegerType(), False),
    T.StructField('campaign_name',  T.StringType(),  False),
    T.StructField('channel',        T.StringType(),  False),
    T.StructField('start_date',     T.StringType(),  False),
    T.StructField('end_date',       T.StringType(),  False),
    T.StructField('budget_eur',     T.DoubleType(),  False),
    T.StructField('year',           T.IntegerType(), False),
])

dim_campaign = spark.createDataFrame(campaign_rows, schema=campaign_schema)
dim_campaign.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .saveAsTable(f'{DATABASE}.dim_campaign')
print(f'dim_campaign: {dim_campaign.count()} rows')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 16, Finished, Available, Finished, False)

dim_campaign: 62 rows


## 7 · `fact_sales` — Vectorised Generation (~500 M rows)

Strategy: generate one row per order-line using `spark.range()`, then derive all columns with Spark functions.  
Year-over-year volume ramp is achieved by generating separate yearly DataFrames and unioning them.

In [14]:
# ── Pre-compute lookup arrays broadcast to workers ────────────────────────

# Current product keys per category (driver-side, then broadcast)
current_products = dim_product.filter('is_current = true') \
    .select('product_key', 'category', 'unit_price_eur', 'supplier_key').collect()

PROD_ALL        = [r.product_key for r in current_products]
PROD_BIKES      = [r.product_key for r in current_products if r.category == 'Complete Bike']
PROD_COMPONENTS = [r.product_key for r in current_products if r.category == 'Component']
PROD_APPAREL    = [r.product_key for r in current_products if r.category == 'Apparel']
PROD_SERVICES   = [r.product_key for r in current_products if r.category == 'Service']

# Price lookup map  product_key -> unit_price_eur
PRICE_MAP = {r.product_key: r.unit_price_eur for r in current_products}

# Date keys per year (driver-side)
date_rows_coll = dim_date.select('date_key', 'year', 'month_demand_weight').collect()
DATE_BY_YEAR = {}
for dr in date_rows_coll:
    DATE_BY_YEAR.setdefault(dr.year, []).append((dr.date_key, dr.month_demand_weight))

CUST_KEYS  = list(range(1, N_CUSTOMERS * 2, 2))   # odd keys = SCD2 v1
CUST_KEYS2 = list(range(2, N_CUSTOMERS * 2 + 1, 2))  # even = v2

# Currency by delivery country
COUNTRY_CURRENCY = {
    'FR':'EUR','BE':'EUR','NL':'EUR','ES':'EUR','IT':'EUR','PT':'EUR',
    'DE':'EUR','SE':'SEK','NO':'NOK','DK':'DKK','PL':'PLN',
    'CH':'CHF','GB':'GBP'
}
# Approximate FX to EUR (mid-2023 rates used as static proxy)
FX_RATES = {'EUR':1.0,'GBP':0.87,'CHF':0.97,'SEK':11.2,'NOK':11.6,'DKK':7.45,'PLN':4.53}

COUNTRIES_W = [
    ('FR',0.55),('GB',0.10),('DE',0.08),('BE',0.04),('NL',0.04),
    ('ES',0.04),('IT',0.04),('CH',0.03),('SE',0.02),('NO',0.02),
    ('PL',0.02),('PT',0.01),('DK',0.01)
]

print('Lookup arrays ready.')
print(f'  Total current SKUs: {len(PROD_ALL)} | Bikes: {len(PROD_BIKES)} | '
      f'Components: {len(PROD_COMPONENTS)} | Apparel: {len(PROD_APPAREL)} | Services: {len(PROD_SERVICES)}')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 17, Finished, Available, Finished, False)

Lookup arrays ready.
  Total current SKUs: 96 | Bikes: 3 | Components: 14 | Apparel: 72 | Services: 4


In [15]:
# ── Yearly volume schedule ────────────────────────────────────────────────
year_volumes = {}
for i, yr in enumerate(range(2021, 2027)):
    vol = int(BASE_LINES_Y1 * ((1 + CAGR) ** i))
    year_volumes[yr] = vol
    print(f'  {yr}: {vol:,} order lines')

total_lines = sum(year_volumes.values())
print(f'  ── Total: {total_lines:,}')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 18, Finished, Available, Finished, False)

  2021: 1,000,000 order lines
  2022: 1,280,000 order lines
  2023: 1,638,400 order lines
  2024: 2,097,152 order lines
  2025: 2,684,354 order lines
  2026: 3,435,973 order lines
  ── Total: 12,135,879


In [16]:
# ──────────────────────────────────────────────────────────────────────────
#  CORE GENERATION FUNCTION
#  Uses spark.range() for parallelism; all column derivations are Spark SQL
#  expressions — no Python UDFs for the hot path.
# ──────────────────────────────────────────────────────────────────────────

# Broadcast lookup tables
bc_prod_all    = spark.sparkContext.broadcast(PROD_ALL)
bc_prod_bikes  = spark.sparkContext.broadcast(PROD_BIKES)
bc_prod_comp   = spark.sparkContext.broadcast(PROD_COMPONENTS)
bc_prod_app    = spark.sparkContext.broadcast(PROD_APPAREL)
bc_prod_svc    = spark.sparkContext.broadcast(PROD_SERVICES)
bc_price_map   = spark.sparkContext.broadcast(PRICE_MAP)
bc_country_w   = spark.sparkContext.broadcast(COUNTRIES_W)
bc_fx          = spark.sparkContext.broadcast(FX_RATES)
bc_cc          = spark.sparkContext.broadcast(COUNTRY_CURRENCY)

@F.udf(T.StructType([
    T.StructField('date_key',          T.IntegerType(),  False),
    T.StructField('customer_key',      T.IntegerType(),  False),
    T.StructField('product_key',       T.IntegerType(),  False),
    T.StructField('store_key',         T.IntegerType(),  False),
    T.StructField('campaign_key',      T.IntegerType(),  True),
    T.StructField('delivery_country',  T.StringType(),   False),
    T.StructField('currency',          T.StringType(),   False),
    T.StructField('fx_rate_to_eur',    T.DoubleType(),   False),
    T.StructField('quantity',          T.IntegerType(),  False),
    T.StructField('unit_price_local',  T.DoubleType(),   False),
    T.StructField('discount_pct',      T.DoubleType(),   False),
    T.StructField('net_revenue_eur',   T.DoubleType(),   False),
    T.StructField('channel',           T.StringType(),   False),
    T.StructField('is_promo',          T.BooleanType(),  False),
]))
def generate_line(row_id: int, year: int):
    import random as _rng
    r = _rng.Random(row_id * 6364136223846793005 + 1442695040888963407)  # LCG mixing

    prod_all  = bc_prod_all.value
    prod_bike = bc_prod_bikes.value
    prod_comp = bc_prod_comp.value
    prod_app  = bc_prod_app.value
    prod_svc  = bc_prod_svc.value
    price_map = bc_price_map.value
    country_w = bc_country_w.value
    fx_map    = bc_fx.value
    cc_map    = bc_cc.value

    # ── Category mix (bikes rare, components most frequent)
    cat_roll = r.random()
    if cat_roll < 0.04 and prod_bike:
        pk = r.choice(prod_bike)
    elif cat_roll < 0.45 and prod_comp:
        pk = r.choice(prod_comp)
    elif cat_roll < 0.85 and prod_app:
        pk = r.choice(prod_app)
    elif prod_svc:
        pk = r.choice(prod_svc)
    else:
        pk = r.choice(prod_all)

    base_price = price_map.get(pk, 100.0)

    # ── Date key — weighted random within year
    # Simplified: use row_id mod to spread across 365 days uniformly;
    # seasonality weight applied as a multiplier in fact_sales via join with dim_date
    year_start_doy = 0  # offset
    days_in_year   = 365 + (1 if year % 4 == 0 else 0)
    doy            = r.randint(1, days_in_year)

    # Convert year + day-of-year to YYYYMMDD integer
    import datetime
    d = datetime.date(year, 1, 1) + datetime.timedelta(days=doy - 1)
    # Clamp to data range
    d = max(datetime.date(2021, 2, 1), min(d, datetime.date(2026, 2, 27)))
    date_key = int(d.strftime('%Y%m%d'))

    # ── Customer
    cust_key = r.randint(1, 160_000)  # covers both SCD2 versions

    # ── Channel & store
    channel   = 'in_store' if r.random() < 0.25 else 'web'
    store_key = 1 if channel == 'in_store' else 2

    # ── Country & currency
    countries = [c for c, _ in country_w]
    weights   = [w for _, w in country_w]
    country   = r.choices(countries, weights=weights, k=1)[0]
    currency  = cc_map.get(country, 'EUR')
    fx        = fx_map.get(currency, 1.0)

    # ── Campaign (30% chance of attributed campaign)
    camp_key = r.randint(1, 60) if r.random() < 0.30 else None

    # ── Quantity (apparel/accessories can be >1)
    qty = r.choices([1, 2, 3], weights=[80, 15, 5], k=1)[0]

    # ── Discount
    is_promo = r.random() < 0.18
    if is_promo:
        disc = round(r.choice([0.05, 0.10, 0.15, 0.20, 0.25, 0.30]), 2)
    else:
        disc = 0.0

    unit_price_local = round(base_price / fx * r.uniform(0.98, 1.02), 2)
    net_rev_eur      = round(unit_price_local * fx * qty * (1 - disc), 2)

    return (date_key, cust_key, pk, store_key, camp_key,
            country, currency, fx, qty, unit_price_local,
            disc, net_rev_eur, channel, is_promo)

print('UDF registered.')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 19, Finished, Available, Finished, False)

UDF registered.


In [17]:
# ── Generate fact_sales year by year and write incrementally ──────────────
# This avoids OOM on large clusters by not holding all years in memory.

# Load the demand-weight table for post-generation reweighting (optional analytics)
dim_date_broadcast = spark.table(f'{DATABASE}.dim_date') \
    .select('date_key', 'month_demand_weight', 'is_race_event_window', 'year_month') \
    .hint('broadcast')

first_year = True
cumulative = 0

for yr, n_lines in year_volumes.items():
    print(f'  Generating {yr}: {n_lines:,} lines...')

    df_year = spark.range(cumulative, cumulative + n_lines, 1, numPartitions=400) \
        .withColumnRenamed('id', 'row_id') \
        .withColumn('year', F.lit(yr)) \
        .withColumn('line_data', generate_line(F.col('row_id'), F.col('year'))) \
        .select(
            F.col('row_id').alias('order_line_id'),
            # Derive a synthetic order_id: group ~3-4 lines per order
            F.floor(F.col('row_id') / F.lit(3)).cast(T.LongType()).alias('order_id'),
            F.col('line_data.date_key'),
            F.col('line_data.customer_key'),
            F.col('line_data.product_key'),
            F.col('line_data.store_key'),
            F.col('line_data.campaign_key'),
            F.col('line_data.delivery_country'),
            F.col('line_data.currency'),
            F.col('line_data.fx_rate_to_eur'),
            F.col('line_data.quantity'),
            F.col('line_data.unit_price_local'),
            F.col('line_data.discount_pct'),
            F.col('line_data.net_revenue_eur'),
            F.col('line_data.channel'),
            F.col('line_data.is_promo'),
            F.lit(yr).alias('year'),
        ) \
        .join(dim_date_broadcast, 'date_key', 'left') \
        .withColumn(
            # Apply demand weight: resample net_revenue stochastically
            'net_revenue_eur',
            F.round(
                F.col('net_revenue_eur') * F.col('month_demand_weight'), 2
            )
        ) \
        .drop('month_demand_weight')

    write_mode = 'overwrite' if first_year else 'append'
    df_year.write.format('delta') \
        .mode(write_mode) \
        .option('overwriteSchema', 'true') \
        .partitionBy('year_month') \
        .saveAsTable(f'{DATABASE}.fact_sales')

    first_year = False
    cumulative += n_lines
    print(f'    → Written. Cumulative lines so far: {cumulative:,}')

print(f'\nfact_sales generation complete. Total target: {total_lines:,}')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 20, Finished, Available, Finished, False)

  Generating 2021: 1,000,000 lines...
    → Written. Cumulative lines so far: 1,000,000
  Generating 2022: 1,280,000 lines...
    → Written. Cumulative lines so far: 2,280,000
  Generating 2023: 1,638,400 lines...
    → Written. Cumulative lines so far: 3,918,400
  Generating 2024: 2,097,152 lines...
    → Written. Cumulative lines so far: 6,015,552
  Generating 2025: 2,684,354 lines...
    → Written. Cumulative lines so far: 8,699,906
  Generating 2026: 3,435,973 lines...
    → Written. Cumulative lines so far: 12,135,879

fact_sales generation complete. Total target: 12,135,879


## 8 · `fact_returns`

In [18]:
# ── Sample ~4 % of sales lines as return events ───────────────────────────
# Read fact_sales in micro-batches using native Delta sampling
fact_sales_df = spark.table(f'{DATABASE}.fact_sales')

returns_df = fact_sales_df \
    .sample(withReplacement=False, fraction=0.04, seed=RANDOM_SEED + 10) \
    .select(
        F.col('order_line_id').alias('original_order_line_id'),
        F.col('date_key'),
        F.col('product_key'),
        F.col('customer_key'),
        F.col('net_revenue_eur').alias('original_net_revenue_eur'),
        F.col('currency'),
        F.col('fx_rate_to_eur'),
        F.col('channel'),
        F.col('year_month'),
    ) \
    .withColumn('return_id',
        F.monotonically_increasing_id()) \
    .withColumn('return_date_key',
        # Return happens 3-30 days after sale — offset date_key by random integer
        (F.col('date_key') +
         F.abs(F.hash(F.col('return_id'))).cast(T.IntegerType()) % 27 + 3
        ).cast(T.IntegerType())) \
    .withColumn('return_reason_key',
        (F.abs(F.hash(F.col('return_id') * 3)).cast(T.IntegerType()) % 6 + 1)) \
    .withColumn('refund_amount_eur',
        F.round(F.col('original_net_revenue_eur') * F.when(
            F.col('return_reason_key') == 2, 1.0   # defective = full refund
        ).when(
            F.col('return_reason_key') == 4, 1.10  # damaged = + return shipping
        ).otherwise(
            F.lit(rng.uniform(0.85, 1.0))           # partial for others
        ), 2)) \
    .withColumn('return_channel',
        F.when(F.col('channel') == 'in_store', 'in_store').otherwise('mail')) \
    .select(
        'return_id', 'original_order_line_id', 'return_date_key',
        'return_reason_key', 'product_key', 'customer_key',
        'currency', 'refund_amount_eur', 'return_channel', 'year_month'
    )

returns_df.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .partitionBy('year_month') \
    .saveAsTable(f'{DATABASE}.fact_returns')

print(f'fact_returns: ~{returns_df.count():,} rows')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 21, Finished, Available, Finished, False)

fact_returns: ~485,039 rows


## 9 · `fact_inventory`

In [19]:
# ── Daily inventory snapshot: 400 SKUs × 2 locations × 1826 days ──────────
# Strategy: cross-join date × product × store, then compute stock movements
# using cumulative window functions (no Python loops).

current_products_df = spark.table(f'{DATABASE}.dim_product').filter('is_current = true') \
    .select('product_key', 'category', 'unit_price_eur')

dates_df = spark.table(f'{DATABASE}.dim_date') \
    .select('date_key', 'full_date', 'year_month')

stores_df = spark.table(f'{DATABASE}.dim_store') \
    .select('store_key', 'store_type')

# Cross join skeleton
skeleton = dates_df.crossJoin(current_products_df).crossJoin(stores_df)

# Aggregate daily sales from fact_sales
daily_sales = spark.table(f'{DATABASE}.fact_sales') \
    .groupBy('date_key', 'product_key', 'store_key') \
    .agg(F.sum('quantity').alias('units_sold'))

# Aggregate daily returns
daily_returns = spark.table(f'{DATABASE}.fact_returns') \
    .withColumnRenamed('return_date_key', 'date_key') \
    .groupBy('date_key', 'product_key') \
    .agg(F.count('return_id').alias('units_returned'))

# Build inventory table
inventory_raw = skeleton \
    .join(daily_sales, ['date_key', 'product_key', 'store_key'], 'left') \
    .join(daily_returns, ['date_key', 'product_key'], 'left') \
    .withColumn('units_sold',     F.coalesce('units_sold',     F.lit(0))) \
    .withColumn('units_returned', F.coalesce('units_returned', F.lit(0))) \
    .withColumn('units_received',
        # Simulate weekly replenishment:  every 7 days receive stock
        F.when(
            (F.dayofweek(F.col('full_date')) == 2) &  # Monday
            (F.col('store_type') == 'Physical'),
            F.greatest(
                F.lit(5),
                (F.col('units_sold') * 7 * F.lit(1.2)).cast(T.IntegerType())
            )
        ).when(
            (F.dayofweek(F.col('full_date')).isin([2, 4])) &  # Mon + Wed
            (F.col('store_type') == 'Online'),
            F.greatest(
                F.lit(20),
                (F.col('units_sold') * 3 * F.lit(1.3)).cast(T.IntegerType())
            )
        ).otherwise(F.lit(0))
    )

# Cumulative stock using window
win = Window.partitionBy('product_key', 'store_key').orderBy('date_key') \
           .rowsBetween(Window.unboundedPreceding, Window.currentRow)

inventory_df = inventory_raw \
    .withColumn('net_movement',
        F.col('units_received') + F.col('units_returned') - F.col('units_sold')) \
    .withColumn('opening_stock_raw',
        F.sum('net_movement').over(win) - F.col('net_movement')) \
    .withColumn('opening_stock',  F.greatest(F.lit(0), F.col('opening_stock_raw'))) \
    .withColumn('closing_stock',  F.greatest(F.lit(0),
        F.col('opening_stock') + F.col('net_movement'))) \
    .withColumn('stock_value_eur',
        F.round(F.col('closing_stock') * F.col('unit_price_eur'), 2)) \
    .select(
        'date_key', 'product_key', 'store_key', 'year_month',
        'opening_stock', 'units_received', 'units_sold',
        'units_returned', 'closing_stock', 'stock_value_eur'
    )

inventory_df.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .partitionBy('year_month') \
    .saveAsTable(f'{DATABASE}.fact_inventory')

print(f'fact_inventory: ~{inventory_df.count():,} rows')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 22, Finished, Available, Finished, False)

fact_inventory: ~533,664 rows


## 10 · `fact_marketing`

In [20]:
# ── One row per campaign × day within its active window ───────────────────
from pyspark.sql.functions import expr

campaigns_df  = spark.table(f'{DATABASE}.dim_campaign')
dates_s_df    = spark.table(f'{DATABASE}.dim_date').select('date_key', 'full_date', 'year_month')

# Expand campaign date ranges to daily rows
marketing_df = campaigns_df \
    .join(dates_s_df,
          (F.col('full_date') >= F.col('start_date')) &
          (F.col('full_date') <= F.col('end_date')), 'inner') \
    .withColumn('duration_days',
        F.datediff(F.col('end_date'), F.col('start_date')) + 1) \
    .withColumn('daily_budget', F.round(F.col('budget_eur') / F.col('duration_days'), 2)) \
    .withColumn('impressions',
        (F.col('daily_budget') *
         F.when(F.col('channel') == 'display',      F.lit(150))
          .when(F.col('channel') == 'paid_social',   F.lit(80))
          .when(F.col('channel') == 'influencer',    F.lit(40))
          .otherwise(F.lit(20))                       # email
        ).cast(T.LongType())) \
    .withColumn('ctr',
        F.when(F.col('channel') == 'email',          F.lit(0.22))
         .when(F.col('channel') == 'paid_social',    F.lit(0.035))
         .when(F.col('channel') == 'display',        F.lit(0.008))
         .otherwise(F.lit(0.06))) \
    .withColumn('clicks',
        (F.col('impressions') * F.col('ctr')).cast(T.LongType())) \
    .withColumn('conversion_rate', F.lit(0.018)) \
    .withColumn('attributed_orders',
        (F.col('clicks') * F.col('conversion_rate')).cast(T.LongType())) \
    .withColumn('avg_order_value', F.lit(220.0)) \
    .withColumn('attributed_revenue_eur',
        F.round(F.col('attributed_orders') * F.col('avg_order_value'), 2)) \
    .withColumn('spend_eur', F.col('daily_budget')) \
    .select(
        'date_key', 'campaign_key', 'campaign_name', 'channel', 'year_month',
        'impressions', 'clicks', 'spend_eur',
        'attributed_orders', 'attributed_revenue_eur'
    )

marketing_df.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
    .partitionBy('year_month') \
    .saveAsTable(f'{DATABASE}.fact_marketing')

print(f'fact_marketing: {marketing_df.count():,} rows')

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 23, Finished, Available, Finished, False)

fact_marketing: 1,161 rows


## 11 · Delta Optimisation

In [17]:
# ── OPTIMIZE + ZORDER for query-critical fact tables ─────────────────────
# Note: OPTIMIZE is a Databricks Delta feature; on OSS Delta it is a no-op
#       if not supported — the tables remain valid.

tables_to_optimize = {
    'fact_sales':      'date_key, product_key',
    'fact_returns':    'return_date_key, product_key',
    'fact_inventory':  'date_key, product_key',
    'fact_marketing':  'date_key, campaign_key',
}

for table, zorder_cols in tables_to_optimize.items():
    try:
        spark.sql(f'OPTIMIZE {DATABASE}.{table} ZORDER BY ({zorder_cols})')
        print(f'  OPTIMIZE {table} done.')
    except Exception as e:
        print(f'  OPTIMIZE {table} skipped ({type(e).__name__}: {str(e)[:80]})')

# ── Analyse all tables (collect statistics for the query planner)
all_tables = [
    'dim_date', 'dim_product', 'dim_customer', 'dim_store',
    'dim_supplier', 'dim_campaign', 'dim_return_reason',
    'fact_sales', 'fact_returns', 'fact_inventory', 'fact_marketing'
]
for t in all_tables:
    spark.sql(f'ANALYZE TABLE {DATABASE}.{t} COMPUTE STATISTICS')
    print(f'  ANALYZE {t} done.')

print('\nAll optimisations applied.')

StatementMeta(, 667c867c-de78-4e99-a7aa-0d46b3775262, 19, Finished, Available, Finished, False)

  OPTIMIZE fact_sales done.
  OPTIMIZE fact_returns done.
  OPTIMIZE fact_inventory done.
  OPTIMIZE fact_marketing done.
  ANALYZE dim_date done.
  ANALYZE dim_product done.
  ANALYZE dim_customer done.
  ANALYZE dim_store done.
  ANALYZE dim_supplier done.
  ANALYZE dim_campaign done.
  ANALYZE dim_return_reason done.
  ANALYZE fact_sales done.
  ANALYZE fact_returns done.
  ANALYZE fact_inventory done.
  ANALYZE fact_marketing done.

All optimisations applied.


## 12 · Verification & Assertions

In [18]:
print('=' * 65)
print('MIMOSA GRAVEL — DATA QUALITY CHECKS')
print('=' * 65)

# ── 1. Row count summary ──────────────────────────────────────────────────
all_tables = [
    'dim_date', 'dim_product', 'dim_customer', 'dim_store',
    'dim_supplier', 'dim_campaign', 'dim_return_reason',
    'fact_sales', 'fact_returns', 'fact_inventory', 'fact_marketing'
]
print('\n── Row counts ──────────────────────────────────────────────')
for t in all_tables:
    cnt = spark.table(f'{DATABASE}.{t}').count()
    print(f'  {t:30s}: {cnt:>15,}')

# ── 2. SCD2 checks ────────────────────────────────────────────────────────
print('\n── SCD2 dim_product ────────────────────────────────────────')
dp = spark.table(f'{DATABASE}.dim_product')
v2_count = dp.groupBy('product_nk').count().filter('count > 1').count()
print(f'  SKUs with >1 SCD2 version: {v2_count:,}')
assert v2_count > 0, 'FAIL: no SCD2 versions found in dim_product'
print('  PASS: SCD2 versions exist.')

print('\n── SCD2 dim_customer ───────────────────────────────────────')
dc = spark.table(f'{DATABASE}.dim_customer')
dc_v2 = dc.groupBy('customer_nk').count().filter('count > 1').count()
print(f'  Customers with >1 SCD2 version: {dc_v2:,}')
assert dc_v2 > 0, 'FAIL: no SCD2 versions in dim_customer'
print('  PASS.')

# ── 3. Seasonality: monthly revenue aggregation ───────────────────────────
print('\n── Monthly revenue (EUR) — seasonality check ───────────────')
monthly = spark.table(f'{DATABASE}.fact_sales') \
    .groupBy('year_month') \
    .agg(F.round(F.sum('net_revenue_eur') / 1e6, 2).alias('rev_M_eur')) \
    .orderBy('year_month') \
    .filter(F.col('year_month').between(202101, 202112))
monthly.show(12, truncate=False)

# ── 4. YoY revenue growth check ───────────────────────────────────────────
print('── Year-over-year revenue growth ───────────────────────────')
yearly = spark.table(f'{DATABASE}.fact_sales') \
    .groupBy('year') \
    .agg(F.round(F.sum('net_revenue_eur') / 1e6, 1).alias('rev_M_eur')) \
    .orderBy('year')
yearly.show(truncate=False)

# ── 5. Returns join rate ──────────────────────────────────────────────────
print('── Returns rate ────────────────────────────────────────────')
n_sales   = spark.table(f'{DATABASE}.fact_sales').count()
n_returns = spark.table(f'{DATABASE}.fact_returns').count()
rate = n_returns / n_sales * 100
print(f'  Sales lines: {n_sales:,} | Returns: {n_returns:,} | Rate: {rate:.2f}%')
assert 2.0 <= rate <= 6.0, f'FAIL: unexpected return rate {rate:.2f}%'
print('  PASS: return rate within 2-6% expected range.')

# ── 6. Inventory — no negative closing stock ──────────────────────────────
print('\n── Inventory negative stock check ──────────────────────────')
neg_stock = spark.table(f'{DATABASE}.fact_inventory') \
    .filter('closing_stock < 0').count()
print(f'  Rows with closing_stock < 0: {neg_stock}')
assert neg_stock == 0, f'FAIL: {neg_stock} rows with negative stock'
print('  PASS: no negative stock.')

# ── 7. Currency distribution in fact_sales ───────────────────────────────
print('\n── Currency distribution ───────────────────────────────────')
spark.table(f'{DATABASE}.fact_sales') \
    .groupBy('currency') \
    .agg(F.round(F.count('*') * 100.0 / n_sales, 2).alias('pct')) \
    .orderBy(F.desc('pct')) \
    .show(truncate=False)

print('\n' + '=' * 65)
print('ALL CHECKS PASSED — Mimosa Gravel Lakehouse is ready!')
print('=' * 65)

StatementMeta(, 667c867c-de78-4e99-a7aa-0d46b3775262, 20, Finished, Available, Finished, False)

MIMOSA GRAVEL — DATA QUALITY CHECKS

── Row counts ──────────────────────────────────────────────
  dim_date                      :           1,853
  dim_product                   :             188
  dim_customer                  :         160,000
  dim_store                     :               3
  dim_supplier                  :              15
  dim_campaign                  :              62
  dim_return_reason             :               6
  fact_sales                    :     728,152,823
  fact_returns                  :      29,129,929
  fact_inventory                :         533,664
  fact_marketing                :           1,161

── SCD2 dim_product ────────────────────────────────────────
  SKUs with >1 SCD2 version: 92
  PASS: SCD2 versions exist.

── SCD2 dim_customer ───────────────────────────────────────
  Customers with >1 SCD2 version: 80,000
  PASS.

── Monthly revenue (EUR) — seasonality check ───────────────
+----------+---------+
|year_month|rev_M_eur|
+---------

In [3]:
%%sql

ALTER TABLE mimosa_gravel.fact_sales SET TBLPROPERTIES("delta.parquet.vorder.enabled" = "true");

OPTIMIZE  mimosa_gravel.fact_sales VORDER

StatementMeta(, c5470704-9ac9-4a25-9634-fadbe35e554c, 6, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 2 fields>